In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx

#==============================================================================
# 1. 全局参数 & H₂ 分子定义
#==============================================================================
K = 2  # NES-VMC 要计算的低激发态数量（基态+1个激发态）
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准（4个态）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# 转为 NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# 1. 定义原始 Hilbert 空间 (同之前)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
# 2. 定义 NES-VMC 参数
K = 2  # 想要同时采样的组态数
# 3. 修正点：使用指数运算符 ** 来构建扩展 Hilbert 空间
hi_ensemble = hi ** K
# 4. 后续的采样器设置保持不变
edges = [(0, 1), (2, 3)] #edges 是对 hi 的 edges
g = nk.graph.Graph(edges=edges)
# hi_ensemble.all_states()[0].shape # (12,)

hi_K = nk.hilbert.TensorHilbert(hi**K)
hi_K.all_states().shape #(4**K,4*K)

single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
rules_tuple = tuple(single_rule for _ in range(K))
tensor_rule = nk.sampler.rules.TensorRule(hilbert=hi_K, rules=[single_rule]*K)

sampler = nk.sampler.MetropolisSampler(hi_ensemble, rule=tensor_rule, n_chains=16, sweep_size=32)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [5]:
hi_ensemble.all_states()[0].reshape(2,4)

Array([[0, 1, 0, 1],
       [0, 1, 0, 1]], dtype=int8)

In [6]:
#==============================================================================
# 3. NES-VMC 神经网络模型（复数 FFNN）
#==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        #log_out = jnp.log(out)
        return jnp.squeeze(out)
    
class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=K, hidden_dim=16):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals
        
        # 1. 生成一串不同的 JAX 随机 key
        key = jax.random.key(42)          # 主种子
        keys = jax.random.split(key, n_states)  # 拆成 n_states 个不同 key

        # 2. 给每个 ansatz 分配不同的 key → 不同的 nnx.Rngs
        # ✅ 这是 NNX 官方支持的写法
        self.single_ansatz_list = [
            SingleStateAnsatz(
                n_spin_orbitals,
                hidden_dim,
                rngs=nnx.Rngs(keys[i])  # 每个用不同 key
            )
            for i in range(n_states)
        ]

    def __call__(self, x_batch):
        '''
        x_batch: (K, n_spin_orbitals)
        '''
        K_state = self.n_states
        M = []
        for i in range(K_state):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K_state)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        log_psi_total = jnp.log(psi_total)
        return psi_total, M
    
K=2
total_ansatz = NESTotalAnsatz(4, n_states=K, hidden_dim=4*3)

In [9]:
import jax
import jax.numpy as jnp

def forward(params, x_batch):
    #print(x_batch.shape)
    # x_batch: (n_chains, chain_length, n_spin_orbitals*K)
    n_chains = x_batch.shape[0]
    K = 2
    n_spin = 4
    # 重塑为 (n_chains, K, n_spin)
    x_reshaped = x_batch.reshape(n_chains, K, n_spin)
    
    # 定义单个联合样本的计算
    def single_logpsi(params, x):
        (psi, _), _ = nnx.call(params)(x)
        return jnp.log(psi)  # 标量复数
    
    # 批量映射
    log_psi_batch = jax.vmap(single_logpsi, in_axes=(None, 0))(params, x_reshaped)
    return log_psi_batch  # 形状 (n_chains,)

parameters = nnx.split(total_ansatz) 
sampler_state = sampler.init_state(forward, parameters, seed=1)
samples, sampler_state = sampler.sample(
    forward, parameters, state=sampler_state, chain_length=500
)
samples.shape

(16, 500, 8)

In [ ]:
#==============================================================================
# 4. 核心：哈密顿作用 + 局部能量矩阵（论文标准）
#==============================================================================
def Ham_psi(ha: nk.operator.DiscreteOperator, single_ansatz:SingleStateAnsatz, x: jnp.array):
    x_primes, mels = ha.get_conn_padded(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_psi(ha: nk.operator.DiscreteOperator, single_ansatz:SingleStateAnsatz, x: jnp.array):
    # 🔥 关键：squeeze 去掉多余维度，确保 x 是 1 维！
    x = x.squeeze()
    # 🔥 关键：用 flattened 版本支持批量/自动处理
    x_primes, mels = ha.get_conn_padded(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)


def Ham_Psi(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch):
    K_state = total_ansatz.n_states
    H_mat = []
    for i in range(K_state):
        row = []
        for j in range(K_state):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha:nk.operator.DiscreteOperator, total_ansatz: NESTotalAnsatz, x_batch):
    """
    x_batch.shape = (K, n_spin_orbitals)
    """
    psi, M = total_ansatz(x_batch)
    eps = 1e-3
    M = M + eps * jnp.eye(M.shape[0], dtype=M.dtype)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M, Hp)
    return jnp.trace(el_mat), el_mat

def compute_local_energy_single(ham: nk.operator.DiscreteOperator, 
                               model: NESTotalAnsatz, 
                               x_batch):
    # x_batch: (K, n_spin_orbitals)
    tr_el, el_mat = compute_local_energy(ham, model, x_batch)
    return tr_el.real, el_mat  # 确保返回实数迹

compute_local_energy_batch = jax.vmap(
    compute_local_energy_single,
    # 对应 4 个入参：ham, model, x_batch, return_log_psi
    in_axes=(None, None, 0),
    out_axes=(0, 0),
    axis_name='samples_batch'
)

def compute_average_local_energy(ham: nk.operator.DiscreteOperator, 
                                 model: NESTotalAnsatz, 
                                 samples, 
                                 ):
    
    '''
    samples.shape = (n_samples, K, n_spin_orbitals)
    '''
    tr_els, el_mats = compute_local_energy_batch(
        ham, model, samples
    )
    
    tr_avg = tr_els.mean()
    el_mat_avg = el_mats.mean(axis=0)
    
    return tr_avg, el_mat_avg

def loss_fn_single(parameters, ham, x_batch):
    # 把参数放回模型
    graph,params = parameters
    model = nnx.merge(graph, params)
    # 计算 loss = trace(local energy matrix)
    tr_el, _ = compute_local_energy_single(ham, model, x_batch)
    return tr_el  # 我们对这个标量求导！

def loss_fn_single(parameter, ham, x_batch):
    # 把参数放回模型
    graph,params = parameters
    model = nnx.merge(graph, params)
    # 计算 loss = trace(local energy matrix)
    tr_el, _ = compute_local_energy_single(ham, model, x_batch)
    return tr_el  # 我们对这个标量求导！


def loss_fn_batch(params,ham, x_batch):
    # 把参数放回模型
    graphdef, variables = params
    model = nnx.merge(graphdef, variables)
    # 计算 loss = trace(local energy matrix)
    tr_el, _ = compute_average_local_energy(ham, model, x_batch)
    return tr_el  # 我们对这个标量求导！




In [ ]:
value_and_grad_single = jax.value_and_grad(loss_fn_single)
value_and_grad_batch = jax.value_and_grad(loss_fn_batch)
# loss, grads = value_and_grad(params, graph, ha, samples)
paramters = nnx.split(total_ansatz)

In [ ]:
compute_local_energy_single(ha, total_ansatz, samples.reshape(-1,K,4)[0])

In [ ]:
loss_fn_single(paramters, ha, samples.reshape(-1,K,4)[0]) #成功
loss_fn_batch(paramters, ha, samples.reshape(-1,K,4)) #成功
value_and_grad_batch(paramters,ha,samples.reshape(-1,K,4)) #成功


In [ ]:
def train_step(params, opt_state, graph, ham, samples):
    # 1. 计算【平均损失】 + 【平均梯度】
    loss, grads = value_and_grad_batch(graph,params, ham, samples)
    
    # 2. 优化器更新参数
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    return params, opt_state, loss

In [ ]:
import optax
# 拆分模型：graph 结构 + params 可训练参数
params,graph = nnx.split(total_ansatz)

# 优化器（Adam 稳定）
optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(params)

In [ ]:
total_ansatz = NESTotalAnsatz(4,2,4*4)
graphdef, variables = nnx.split(total_ansatz)  # ✅ 拆成结构 + 参数
sampler_state = sampler.init_state(forward, (graphdef, variables), seed=1)
# 优化器（Adam 稳定）
optimizer = optax.adam(learning_rate=1e-2)
opt_state = optimizer.init(variables)  # ✅ 只初始化变量！不传 graphdef！
loss_record=[]
for step in range(100):
    sampler_state = sampler.reset(forward, (graphdef, variables), sampler_state)
    samples, sampler_state = sampler.sample(
        forward, (graphdef, variables), state=sampler_state, chain_length=200
    )
    params_for_loss = (graphdef, variables)
    loss, grads = value_and_grad_batch(
        params_for_loss, ha, samples.reshape(-1, K, 4)
    )
    loss_record.append(loss)
    if step % 20 == 0:
        print(f'Trace of Total_ansatz matirx| [{step}] loss: {loss:.8f} ')
    grad_graph, grad_vars = grads
    updates, opt_state = optimizer.update(grad_vars, opt_state, variables)
    variables = optax.apply_updates(variables, updates)
    total_ansatz = nnx.merge(graphdef, variables)

In [ ]:
# 批量采样用于平均能量矩阵
samples_final, _ = sampler.sample(
    forward, parameters, state=sampler_state, chain_length=500
)

# 计算平均局部能量矩阵
el_mat_sum = jnp.zeros((K, K), dtype=complex)
n_sample_final = samples_final.shape[0]

In [ ]:
samples_final.reshape(-1,2,4).shape


In [ ]:
value, M = compute_average_local_energy(ha, total_ansatz, samples_final.reshape(-1,2,4))
eigen_energies = jnp.linalg.eigvalsh(M).real
eigen_energies